# Laboratorium: GeoPandas – część A
**Scenariusz:** odczyty PM2.5 z czujników w Opolu (dane syntetyczne, z błędami).
Pytania: (1) które dzielnice mają najgorsze powietrze, (2) jak jest w pobliżu szkół podstawowych.

## 0. Sprawdzenie środowiska

In [ ]:
import geopandas as gpd
import rasterio
import shapely

print("geopandas", gpd.__version__)
print("rasterio ", rasterio.__version__, "| GDAL", rasterio.__gdal_version__)
print("shapely  ", shapely.__version__)

## A1. Geometria i GeoDataFrame od zera

In [ ]:
from shapely.geometry import Point, Polygon

rynek = Point(17.9226, 50.6681)          # (x, y) = (longitude, latitude)
square = Polygon([(17.92, 50.66), (17.93, 50.66), (17.93, 50.67), (17.92, 50.67)])

print(rynek)
print(square.area)                       # square degrees!
print(square.contains(rynek))

In [ ]:
places = gpd.GeoDataFrame(
    {"name": ["Rynek", "Square"]},
    geometry=[rynek, square],
    crs="EPSG:4326",
)
print(type(places))
print(places.crs)
print(places.geom_type)
places

## A2. Wczytanie surowych danych

In [ ]:
import pandas as pd

raw = pd.read_csv("dane/pm25_sensors.csv", parse_dates=["measured_at"])
print(raw.shape)
raw.head(10)

In [ ]:
raw[["lat", "lon", "pm25"]].describe()

In [ ]:
raw.isna().sum()

In [ ]:
raw.duplicated().sum()

In [ ]:
# 🤔 Skąd min(lat) ≈ 17.9? -> wiersze z zamienioną kolejnością lat/lon
raw[raw["lat"] < 40]

## A3. Czyszczenie danych

In [ ]:
df = raw.copy()
report = {"input rows": len(df)}

# 1. Duplicates
df = df.drop_duplicates()
report["after drop_duplicates"] = len(df)

# 2. Rows without coordinates
df = df.dropna(subset=["lat", "lon"])
report["after dropna"] = len(df)

# 3. Swapped coordinates: in Poland lat is 49-55 and lon is 14-24
swapped = df["lat"].between(14, 24.5) & df["lon"].between(49, 55)
df.loc[swapped, ["lat", "lon"]] = df.loc[swapped, ["lon", "lat"]].to_numpy()
report["swapped rows fixed"] = int(swapped.sum())

# 4. -999 = sensor error code
df = df[df["pm25"] >= 0].reset_index(drop=True)
report["clean rows"] = len(df)

report

In [ ]:
# Eksperyment: zamiana BEZ .to_numpy() - pandas wyrównuje po nazwach kolumn, więc nic się nie zmienia
test = raw.drop_duplicates().dropna(subset=["lat", "lon"]).copy()
sw = test["lat"].between(14, 24.5) & test["lon"].between(49, 55)
test.loc[sw, ["lat", "lon"]] = test.loc[sw, ["lon", "lat"]]
test.loc[sw, ["sensor_id", "lat", "lon"]]

In [ ]:
df[["lat", "lon", "pm25"]].describe().round(2)

## A4. Z tabeli do GeoDataFrame

In [ ]:
sensors = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),   # x = lon, y = lat!
    crs="EPSG:4326",
)
print(sensors.geom_type.unique())
print(sensors.crs)
print(sensors.total_bounds)

In [ ]:
sensors.explore(column="pm25", cmap="Oranges", marker_kwds={"radius": 6})

In [ ]:
# 🤔 Zamienione argumenty points_from_xy -> punkty lądują w okolicach (lon 50.7, lat 17.9): Ocean Indyjski / Somalia
wrong = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lat"], df["lon"]), crs="EPSG:4326")
print(wrong.total_bounds)
wrong.explore()

## A5. Układy współrzędnych

In [ ]:
a, b = sensors.geometry.iloc[0], sensors.geometry.iloc[1]
a.distance(b)          # degrees

In [ ]:
sensors_m = sensors.to_crs("EPSG:2180")
sensors_m.geometry.iloc[0].distance(sensors_m.geometry.iloc[1])   # metres

In [ ]:
sensors_m.head(3)[["sensor_id", "geometry"]]

In [ ]:
from pyproj import Geod

gdansk, zakopane = (18.6466, 54.3520), (19.9496, 49.2992)
pts = gpd.GeoSeries.from_xy([gdansk[0], zakopane[0]], [gdansk[1], zakopane[1]], crs="EPSG:4326")

_, _, true_m = Geod(ellps="WGS84").inv(*gdansk, *zakopane)
print(f"geodesic : {true_m / 1000:8.1f} km")

for crs in ["EPSG:3857", "EPSG:2180"]:
    p = pts.to_crs(crs)
    d = p.iloc[0].distance(p.iloc[1])
    print(f"{crs}: {d / 1000:8.1f} km   error {100 * (d / true_m - 1):+.2f} %")

## A6. Poligony: dzielnice Opola

In [ ]:
districts = gpd.read_file("dane/opole_districts.geojson")
print(len(districts), districts.crs)
districts.head()

In [ ]:
# 🤔 .area w EPSG:4326 -> stopnie kwadratowe + UserWarning "Geometry is in a geographic CRS"
print(f"area in degrees²: {districts.area.sum():.5f}")

In [ ]:
districts = districts.to_crs("EPSG:2180")
districts["area_km2"] = districts.area / 1e6

print(f"City area: {districts['area_km2'].sum():.1f} km²")
districts.sort_values("area_km2", ascending=False).head()

In [ ]:
import matplotlib.pyplot as plt

ax = districts.plot(column="area_km2", cmap="Blues", legend=True,
                    edgecolor="white", figsize=(7, 7))
ax.set_title("Opole districts by area [km²]")
ax.set_axis_off()
plt.show()

## A7. Złączenie przestrzenne

In [ ]:
joined = gpd.sjoin(
    sensors_m,
    districts[["name", "geometry"]],
    how="left",
    predicate="within",
)
print(len(joined), joined["name"].isna().sum())
joined[["sensor_id", "pm25", "name"]].head()

In [ ]:
stats = (joined.groupby("name")
               .agg(pm25_mean=("pm25", "mean"), sensor_count=("pm25", "size"))
               .round(1))

summary = districts.merge(stats, on="name", how="left")
print("districts without sensors:", summary[summary["sensor_count"].isna()]["name"].tolist())
summary.sort_values("pm25_mean", ascending=False).head()

In [ ]:
reliable = summary[summary["sensor_count"] >= 3]
reliable.nlargest(3, "pm25_mean")[["name", "pm25_mean", "sensor_count"]]

In [ ]:
summary.explore(column="pm25_mean", cmap="Oranges", scheme="quantiles", k=5,
                legend_kwds={"caption": "mean PM2.5 [µg/m³]"},
                missing_kwds={"color": "lightgray", "label": "no sensors"})

In [ ]:
# 🤔 within vs intersects - dla punktów leżących wewnątrz poligonów wynik jest identyczny
j2 = gpd.sjoin(sensors_m, districts[["name", "geometry"]], how="left", predicate="intersects")
print(len(j2), (j2["name"].values == joined["name"].values).all())

## A8. Bufory

In [ ]:
schools = gpd.read_file("dane/opole_schools.geojson").to_crs("EPSG:2180")

buffers = schools.copy()
buffers["geometry"] = schools.buffer(500)
buffers.head(3)

In [ ]:
near_schools = gpd.sjoin(sensors_m, buffers[["name", "geometry"]], predicate="within")

print("schools:", len(schools))
print("schools with a sensor within 500 m:", near_schools["name"].nunique())
print(f"mean PM2.5 near schools: {near_schools['pm25'].mean():.1f} µg/m³")

In [ ]:
city = districts.union_all()
coverage = buffers.union_all().intersection(city).area / city.area
print(f"{coverage:.1%} of the city")

In [ ]:
m = districts.explore(color="gray", style_kwds={"fillOpacity": 0.05}, name="districts")
buffers.explore(m=m, color="orange", name="500 m buffers")
sensors_m.explore(m=m, color="black", marker_kwds={"radius": 3}, name="sensors")
m

In [ ]:
# 🤔 Promień 1500 m
buf15 = schools.copy(); buf15["geometry"] = schools.buffer(1500)
ns15 = gpd.sjoin(sensors_m, buf15[["name", "geometry"]], predicate="within")
print("schools with a sensor within 1500 m:", ns15["name"].nunique())
print(f"coverage: {buf15.union_all().intersection(city).area / city.area:.1%}")

# ... i to samo w EPSG:4326: buffer(1500) = 1500 STOPNI -> poligon obejmujący cały glob
bad = gpd.read_file("dane/opole_schools.geojson").buffer(1500)   # UserWarning: geographic CRS
print(bad.iloc[0].bounds)

## A9. Zapis wyników

In [ ]:
from pathlib import Path

out = Path("wyniki")
out.mkdir(exist_ok=True)

summary.to_file(out / "pm25.gpkg", layer="districts", driver="GPKG")
sensors_m.to_file(out / "pm25.gpkg", layer="sensors", driver="GPKG")
summary.to_crs("EPSG:4326").to_file(out / "pm25_districts.geojson", driver="GeoJSON")

gpd.list_layers(out / "pm25.gpkg")

In [ ]:
check = gpd.read_file(out / "pm25.gpkg", layer="districts")
print(len(check), check.crs)